# nb12.3 — MARS Repository Schema: `metadata.json` and `README.md` That Pass Validation

*Week 12, Chapter 12.3.* GMU's Mason Archival Repository Service (MARS) is the
university's long-term home for your data and code. Once you deposit there,
your paper has a persistent handle (a stable URL), a license, and a citation
your future grad-school applications can point to.

MARS accepts deposits only if they conform to a minimal metadata schema. This
notebook builds a `metadata.json` and a companion `README.md` for Priya's
climate-insurance replication packet, validates the JSON against a hand-written
JSON Schema (using `jsonschema` if available; falling back to a hand-rolled
validator otherwise), and produces a deposit-ready bundle.


In [ ]:
import matplotlib
matplotlib.use("Agg")

import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

WORK = Path("/tmp/nb12_3_mars")
WORK.mkdir(exist_ok=True)


## 1. What MARS requires (the short version)

The MARS schema we target in this notebook has eight required top-level fields
and four optional ones. Required fields establish *who, what, when, and how*;
optional fields handle licensing, funding, and related-publication links.

| Field            | Type     | Required | Note |
|------------------|----------|----------|------|
| `title`          | string   | yes      | Paper title; <= 300 characters |
| `authors`        | array    | yes      | At least one author object with `name` and `affiliation` |
| `abstract`       | string   | yes      | 200-4000 characters |
| `keywords`       | array    | yes      | 3-8 strings |
| `subjects`       | array    | yes      | Library of Congress Subject Headings (LCSH) -- at least one |
| `date_created`   | string   | yes      | ISO-8601 date |
| `language`       | string   | yes      | ISO 639-1 (e.g., `"en"`) |
| `resource_type`  | string   | yes      | One of: `"dataset"`, `"software"`, `"text"`, `"replication-package"` |
| `license`        | string   | no       | SPDX identifier (e.g., `"CC-BY-4.0"`) |
| `doi`            | string   | no       | DOI of related publication, if any |
| `funder`         | string   | no       | Grant agency, if any |
| `related_url`    | string   | no       | URL of working paper, conference page, etc. |

The schema we write below encodes all of these as a JSON Schema document.


## 2. The JSON Schema

JSON Schema is a small DSL (draft 2020-12) for declaring what counts as a
valid JSON document. The four primitives we need are: `type` (string, integer,
array, object), `required` (which keys must appear), `minItems`/`maxItems` (for
arrays), and `pattern` (for regex constraints on strings). We assemble the
schema below as a regular Python dict and dump it to disk so MARS reviewers can
audit it.


In [ ]:
MARS_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "$id": "https://mars.gmu.edu/schemas/deposit-2026-v1.json",
    "title": "MARS Deposit Metadata (2026 v1)",
    "type": "object",
    "required": [
        "title", "authors", "abstract", "keywords", "subjects",
        "date_created", "language", "resource_type",
    ],
    "properties": {
        "title": {"type": "string", "minLength": 5, "maxLength": 300},
        "authors": {
            "type": "array",
            "minItems": 1,
            "items": {
                "type": "object",
                "required": ["name", "affiliation"],
                "properties": {
                    "name": {"type": "string", "minLength": 2},
                    "affiliation": {"type": "string", "minLength": 2},
                    "orcid": {
                        "type": "string",
                        "pattern": r"^\d{4}-\d{4}-\d{4}-\d{3}[\dX]$",
                    },
                },
            },
        },
        "abstract": {"type": "string", "minLength": 200, "maxLength": 4000},
        "keywords": {
            "type": "array",
            "minItems": 3,
            "maxItems": 8,
            "items": {"type": "string", "minLength": 2},
        },
        "subjects": {
            "type": "array",
            "minItems": 1,
            "items": {"type": "string", "minLength": 3},
        },
        "date_created": {
            "type": "string",
            "pattern": r"^\d{4}-\d{2}-\d{2}$",
        },
        "language": {"type": "string", "pattern": r"^[a-z]{2}$"},
        "resource_type": {
            "type": "string",
            "enum": ["dataset", "software", "text", "replication-package"],
        },
        "license": {"type": "string"},
        "doi": {"type": "string", "pattern": r"^10\.\d{4,9}/.+$"},
        "funder": {"type": "string"},
        "related_url": {"type": "string", "pattern": r"^https?://"},
    },
    "additionalProperties": False,
}

(WORK / "mars-schema-2026-v1.json").write_text(json.dumps(MARS_SCHEMA, indent=2))
print("schema written;", sum(1 for _ in MARS_SCHEMA["properties"]), "properties defined")


## 3. Priya's metadata

Priya's paper studies how Florida property-insurance premiums respond to
realized hurricane damages. We populate the metadata dict with values that we
know obey the schema. (Soon we will deliberately break it to demonstrate the
validator catches errors.)


In [ ]:
priya_metadata = {
    "title": "Hurricane Realizations and Florida Homeowner Insurance Premiums, 2010-2024",
    "authors": [
        {
            "name": "Priya Shah",
            "affiliation": "George Mason University (Schar Symposium Author)",
            "orcid": "0009-0002-3456-7890",
        },
        {
            "name": "Lei Gao",
            "affiliation": "George Mason University",
            "orcid": "0000-0001-9876-5432",
        },
    ],
    "abstract": (
        "We study how Florida homeowner-insurance premiums adjust to realized hurricane "
        "damages using a county-year panel from 2010 to 2024. A one standard deviation "
        "increase in county-level hurricane-related claim payouts is followed by a 6.8 "
        "percent increase in average premiums over the next two policy renewal cycles. "
        "The pass-through is concentrated in counties with concentrated insurer markets "
        "and is muted in counties where the state's catastrophe reinsurance pool absorbed "
        "a larger share of losses. Estimates are robust to county and year fixed effects "
        "and to clustering at the county level."
    ),
    "keywords": ["insurance", "climate risk", "hurricane", "florida", "pass-through"],
    "subjects": [
        "Insurance--Florida",
        "Hurricane damage--Economic aspects",
        "Climate risk",
    ],
    "date_created": "2026-09-11",
    "language": "en",
    "resource_type": "replication-package",
    "license": "CC-BY-4.0",
    "related_url": "https://ssrn.com/abstract=4123456",
}

# Round-trip the dict through JSON to catch any non-serializable values early
priya_metadata = json.loads(json.dumps(priya_metadata))
list(priya_metadata.keys())


## 4. Validation

We try to use the standard `jsonschema` library if it is installed; otherwise we
fall back to a hand-rolled validator that covers the cases this schema uses.
Both code paths return a list of validation errors -- empty list means valid.


In [ ]:
try:
    import jsonschema
    HAVE_JSONSCHEMA = True
except ImportError:
    HAVE_JSONSCHEMA = False

def validate_with_library(data, schema):
    """Validate using the jsonschema library if available."""
    v = jsonschema.Draft202012Validator(schema)
    out = []
    for e in sorted(v.iter_errors(data), key=lambda e: list(e.path)):
        path = "/".join(map(str, e.absolute_path)) or "<root>"
        out.append(f"{path}: {e.message}")
    return out

def validate_hand_rolled(data, schema, path=""):
    """Hand-rolled JSON Schema validator covering: required, type, enum, pattern,
    minLength, maxLength, minItems, maxItems, items, properties, additionalProperties."""
    errors = []
    t = schema.get("type")
    if t == "object":
        if not isinstance(data, dict):
            return [f"{path or '<root>'}: expected object, got {type(data).__name__}"]
        for req in schema.get("required", []):
            if req not in data:
                errors.append(f"{path or '<root>'}: missing required key '{req}'")
        props = schema.get("properties", {})
        allow_extra = schema.get("additionalProperties", True)
        for k, v in data.items():
            sub_path = f"{path}/{k}" if path else k
            if k in props:
                errors.extend(validate_hand_rolled(v, props[k], sub_path))
            elif not allow_extra:
                errors.append(f"{sub_path}: additional property not allowed")
    elif t == "array":
        if not isinstance(data, list):
            return [f"{path}: expected array, got {type(data).__name__}"]
        if "minItems" in schema and len(data) < schema["minItems"]:
            errors.append(f"{path}: array has {len(data)} items, min {schema['minItems']}")
        if "maxItems" in schema and len(data) > schema["maxItems"]:
            errors.append(f"{path}: array has {len(data)} items, max {schema['maxItems']}")
        if "items" in schema:
            for i, item in enumerate(data):
                errors.extend(validate_hand_rolled(item, schema["items"], f"{path}[{i}]"))
    elif t == "string":
        if not isinstance(data, str):
            return [f"{path}: expected string, got {type(data).__name__}"]
        if "minLength" in schema and len(data) < schema["minLength"]:
            errors.append(f"{path}: string length {len(data)} below min {schema['minLength']}")
        if "maxLength" in schema and len(data) > schema["maxLength"]:
            errors.append(f"{path}: string length {len(data)} above max {schema['maxLength']}")
        if "pattern" in schema and not re.search(schema["pattern"], data):
            errors.append(f"{path}: '{data[:40]}' does not match pattern {schema['pattern']}")
        if "enum" in schema and data not in schema["enum"]:
            errors.append(f"{path}: '{data}' not in {schema['enum']}")
    return errors

def validate(data, schema):
    if HAVE_JSONSCHEMA:
        return validate_with_library(data, schema)
    return validate_hand_rolled(data, schema)

print("jsonschema library available:", HAVE_JSONSCHEMA)
errors = validate(priya_metadata, MARS_SCHEMA)
print("Priya metadata errors:", errors if errors else "(none - valid)")


## 5. Deliberate breakage (so we know the validator has teeth)

We make three copies of Priya's metadata, each broken in a different
realistic way, and confirm the validator catches each one with a useful error.


In [ ]:
# Broken copy 1: ORCID with wrong checksum length
broken_orcid = json.loads(json.dumps(priya_metadata))
broken_orcid["authors"][0]["orcid"] = "0009-0002-345"
errors_1 = validate(broken_orcid, MARS_SCHEMA)

# Broken copy 2: only two keywords (minimum is 3)
broken_kw = json.loads(json.dumps(priya_metadata))
broken_kw["keywords"] = ["insurance", "hurricane"]
errors_2 = validate(broken_kw, MARS_SCHEMA)

# Broken copy 3: resource_type that is not on the allowed list
broken_rt = json.loads(json.dumps(priya_metadata))
broken_rt["resource_type"] = "essay"
errors_3 = validate(broken_rt, MARS_SCHEMA)

pd.DataFrame({
    "case": ["bad ORCID", "too few keywords", "unknown resource_type"],
    "n_errors": [len(errors_1), len(errors_2), len(errors_3)],
    "first_error": [errors_1[0] if errors_1 else "",
                    errors_2[0] if errors_2 else "",
                    errors_3[0] if errors_3 else ""],
})


Each broken metadata produces at least one error and the messages are
specific enough that you could fix the problem without re-reading the schema.
That is the contract a validator should deliver.


## 6. Auto-generating `README.md` from the metadata

The MARS deposit also requires a human-readable README. Rather than writing
both files by hand and risking drift between them, we *generate* `README.md`
from the validated metadata. If the metadata is the source of truth, the README
is just a rendering.


In [ ]:
README_TEMPLATE = '# {title}\n\n## Authors\n\n{authors_block}\n\n## Abstract\n\n{abstract}\n\n## Keywords\n\n{keywords_block}\n\n## Subjects\n\n{subjects_block}\n\n## Resource type\n\n`{resource_type}`\n\n## Date created\n\n{date_created}\n\n## License\n\n{license_block}\n\n## Related URL\n\n{related_url_block}\n\n## How to cite\n\nCite this MARS deposit using the persistent handle MARS will assign upon\nacceptance. Until then, cite the related working paper at the URL above.\n\n## Replication notes\n\nThis package is organized following the AEA Data Editor template:\n\n- /data/ -- synthetic input data; see /data/README.md for variable definitions.\n- /code/ -- numbered scripts (01-clean.py, 02-build-panel.py, ...) that run end-to-end.\n- /output/ -- figures and tables exactly as they appear in the paper.\n- environment.yml -- pinned conda environment.\n- manifest.csv -- SHA-256 hashes of every file in the deposit.\n\n## Contact\n\nQuestions: priya.shah@example.edu (please use a stable, non-student-mailbox address).\n'

def render_readme(meta):
    authors_block = "\n".join([
        f"- **{a['name']}** ({a['affiliation']})" +
        (f" -- ORCID `{a['orcid']}`" if "orcid" in a else "")
        for a in meta["authors"]
    ])
    keywords_block = ", ".join(meta["keywords"])
    subjects_block = "\n".join(f"- {s}" for s in meta["subjects"])
    return README_TEMPLATE.format(
        title=meta["title"],
        authors_block=authors_block,
        abstract=meta["abstract"],
        keywords_block=keywords_block,
        subjects_block=subjects_block,
        resource_type=meta["resource_type"],
        date_created=meta["date_created"],
        license_block=meta.get("license", "Not specified"),
        related_url_block=meta.get("related_url", "Not specified"),
    )

readme_text = render_readme(priya_metadata)
print(readme_text[:600], "...")


## 7. Writing the deposit bundle

Three files: `metadata.json`, `README.md`, and a copy of `mars-schema-2026-v1.json`
so that any reviewer who wants to re-validate the deposit has the schema right
there. We then verify our own bundle by reading it back from disk and revalidating.


In [ ]:
def write_deposit(meta, out_dir):
    """Write metadata.json, README.md, and schema copy. Re-validate from disk."""
    out_dir.mkdir(exist_ok=True)
    (out_dir / "metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    (out_dir / "README.md").write_text(render_readme(meta), encoding="utf-8")
    (out_dir / "mars-schema-2026-v1.json").write_text(
        json.dumps(MARS_SCHEMA, indent=2), encoding="utf-8"
    )
    # Round-trip check: read metadata back, revalidate
    read_back = json.loads((out_dir / "metadata.json").read_text())
    return {
        "files": sorted(p.name for p in out_dir.iterdir()),
        "errors_on_reread": validate(read_back, MARS_SCHEMA),
        "metadata_sha256": hashlib.sha256(
            (out_dir / "metadata.json").read_bytes()
        ).hexdigest(),
    }

deposit_report = write_deposit(priya_metadata, WORK / "deposit")
deposit_report


## 8. The subject-heading sanity check

LCSH (Library of Congress Subject Headings) is the controlled vocabulary MARS
indexes against. The schema only enforces "at least one string, length >= 3" --
it cannot verify that your subject heading actually exists in LCSH. We add a
soft check that compares your subjects against a hand-curated short list of
commonly-used LCSH headings for finance and economics papers and flags
suspiciously novel-looking strings.


In [ ]:
LCSH_FINANCE_SHORTLIST = {
    "Banks and banking", "Capital market", "Climate change", "Climate risk",
    "Consumer credit", "Corporate governance", "Credit ratings",
    "Cryptocurrencies", "Discrimination in financial services",
    "Economic indicators", "Finance", "Financial crises",
    "Financial institutions", "Fixed income securities",
    "Foreign exchange rates", "Hurricane damage--Economic aspects",
    "Income tax", "Insurance", "Insurance--Florida", "Investments",
    "Microfinance", "Mortgage loans", "Patents--Economic aspects",
    "Portfolio management", "Risk management",
    "Securities--Government policy", "Stock exchanges",
    "Stocks--Prices", "Venture capital",
}

def check_lcsh(subjects):
    return pd.DataFrame({
        "subject": subjects,
        "in_shortlist": [s in LCSH_FINANCE_SHORTLIST for s in subjects],
    })

check_lcsh(priya_metadata["subjects"])


Two of Priya's three subjects match common LCSH entries; one does not. Either
that one needs to be checked against the real LCSH (id.loc.gov/authorities/subjects)
or it is a perfectly legitimate heading we just did not stock. The check is
*soft* by design: false positives on novel subjects are an acceptable cost for
catching outright typos.


## 9. When it fails

**Schema drift.** MARS will update the schema. Pin the schema version inside
the deposit (we do -- see `$id`) so a future reviewer can resolve any
validation failure to the correct version.

**Hand-rolled validator gaps.** Our fallback validator covers the schema we
wrote and would not be safe for arbitrary user-supplied schemas. If you extend
`MARS_SCHEMA` with features like `oneOf` or `$ref`, install `jsonschema`.

**ORCID typos.** The most common breakage is one missing digit in the ORCID.
The pattern catches it but the error message is dense. A nicer UX (left as a
your-turn exercise) would print "ORCIDs are 16 digits with hyphens after the
4th, 8th, and 12th. Did you mean 0009-0002-3456-7890?"


## 10. Your turn

1. **Add a fifth author.** Add a co-author with an invalid ORCID. Confirm the
   validator catches it and points to the right author index.

2. **Tighten the keywords.** Add a `uniqueItems: true` constraint to the
   `keywords` schema and confirm a duplicate keyword now fails validation.

3. **Round-trip the README.** Write `parse_readme(text)` that pulls the title,
   authors, and date out of a rendered README. Confirm that
   `parse_readme(render_readme(m)) == m` for the relevant fields.

4. **Build Sam's deposit.** Build a `metadata.json` for Sam's intraday-momentum
   replication packet. Confirm validation passes on the first try (it will not;
   that's the point).
